# Understanding Scikit-learn: From Raw Data to Trained Models

**Scikit-learn** is Python's core library for classical machine learning, and it quietly
underpins an enormous share of real-world data science: preprocessing pipelines, baseline
models, evaluation metrics, and hyperparameter search, even in projects whose final model is
a deep neural network built in TensorFlow/Keras or PyTorch. This notebook answers two
questions: **what is scikit-learn**, and **why does it matter enough that almost every
Python machine learning project touches it somewhere?**

It's written as a **learning exercise**, not just a reference: every step includes
plain-English explanations of *what* the code does and *why*, backed by worked examples that
prove the point numerically rather than just asserting it — the same approach used in the
companion `NumPy_Fundamentals` notebook, which this one deliberately reuses a dataset from in
its capstone.

### What we'll do, step by step:
1. Import scikit-learn and see what it gives us
2. Discover the problem scikit-learn solves, by writing a classifier by hand first
3. Understand the core API design: Estimators, Transformers, Predictors
4. Load and split a real dataset
5. See why a uniform API matters: swapping algorithms with almost no code changes
6. Understand *why* preprocessing matters, by breaking a model on purpose
7. Train and evaluate a classical model properly
8. Understand why a single train/test split isn't trustworthy on its own
9. Learn what `Pipeline` is for, and watch a data-leakage bug inflate accuracy on pure noise
10. Automate hyperparameter search, and check it agrees with a hand-rolled version
11. **Capstone**: train a real neural network with scikit-learn on the same handwritten
    digit images used in `NumPy_Fundamentals`, and compare it against several classical models
12. See how scikit-learn fits alongside deep learning frameworks in real projects


## Step 1: Import scikit-learn

By convention, scikit-learn's individual tools are imported directly from their submodules
(e.g. `from sklearn.linear_model import LogisticRegression`) rather than importing the whole
package under a short alias, unlike NumPy's `np.` convention. We'll still import the top-level
package once just to check the version.


In [ ]:
import sklearn
import numpy as np
import time

print("scikit-learn version:", sklearn.__version__)
print("numpy version:", np.__version__)

## Step 2: The problem scikit-learn solves

Let's write one of the simplest machine learning algorithms — **k-nearest neighbors
(k-NN)** — entirely by hand first. The idea is simple: to classify a new point, find the `k`
training points closest to it, and take a majority vote of their labels.

Simple to *describe*, but even this "simple" algorithm has several details to get right:
computing distances to every training point, sorting them, breaking ties, and handling more
than one feature. Now imagine doing this — correctly, efficiently, and bug-free — for a
support vector machine, a random forest, or a neural network, and then *also* building
consistent cross-validation, hyperparameter tuning, and preprocessing around each one. This is
the tedium scikit-learn exists to eliminate.


In [ ]:
from sklearn.datasets import make_classification

# A small synthetic 2D dataset, so the by-hand version stays easy to follow
X_demo, y_demo = make_classification(
    n_samples=30, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, n_classes=2, random_state=7,
)
X_demo_train, X_demo_test = X_demo[:24], X_demo[24:]
y_demo_train, y_demo_test = y_demo[:24], y_demo[24:]


def manual_knn_predict(X_train, y_train, X_query, k=3):
    # Classify each query point by a majority vote of its k nearest training points
    predictions = []
    for query_point in X_query:
        # Euclidean distance from this query point to every training point at once
        distances = np.sqrt(((X_train - query_point) ** 2).sum(axis=1))
        nearest_indices = np.argsort(distances)[:k]
        nearest_labels = y_train[nearest_indices]
        values, counts = np.unique(nearest_labels, return_counts=True)
        predictions.append(values[np.argmax(counts)])
    return np.array(predictions)


manual_predictions = manual_knn_predict(X_demo_train, y_demo_train, X_demo_test, k=3)
print("Manual k-NN predictions:", manual_predictions)
print("Actual labels:          ", y_demo_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

sklearn_knn = KNeighborsClassifier(n_neighbors=3)
sklearn_knn.fit(X_demo_train, y_demo_train)
sklearn_predictions = sklearn_knn.predict(X_demo_test)

print("scikit-learn predictions:", sklearn_predictions)
print("Identical to our manual version:", np.array_equal(manual_predictions, sklearn_predictions))

Both versions agree exactly — our manual loop and scikit-learn's `KNeighborsClassifier`
are doing the same underlying maths. The difference is that scikit-learn's version is
already tested, optimised, handles edge cases we didn't think about, and — as we'll see next
— exposes the *exact same interface* as every other algorithm in the library, so swapping
k-NN for something completely different takes one line, not a rewrite.


## Step 3: The core API — Estimators, Transformers, Predictors

Almost everything in scikit-learn follows the same pattern, called the **Estimator API**:

| Method | What it does |
|---|---|
| `fit(X, y)` | Learn parameters from training data (`y` is omitted for unsupervised models) |
| `predict(X)` | Use the learned model to make predictions on new data |
| `transform(X)` | Apply a learned transformation (used by preprocessing/feature objects) |
| `fit_transform(X)` | Shortcut for calling `fit` then `transform` |
| `score(X, y)` | Return a default performance metric for the model |

Because a `StandardScaler`, a `LogisticRegression`, and a `RandomForestClassifier` all expose
this same interface, code written against one estimator usually works against another with
almost no changes — which is exactly what we'll demonstrate in Step 5.


## Step 4: Loading and splitting a real dataset

We'll use the classic **Iris dataset** (built into scikit-learn) for the next few steps —
150 flower measurements across 3 species, with 4 numeric features per flower.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

print("Feature names:", iris.feature_names)
print("Target classes:", iris.target_names)
print("X shape:", X.shape, " y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:    ", X_test.shape[0])

## Step 5: Why the API matters — swapping algorithms freely

Because every classifier shares the same `fit`/`predict` interface, we can try several
completely different algorithms on the same data using identical code, just by changing
which class we instantiate. (Iris's four measurements are all in centimetres and on
comparable scales, so we can skip preprocessing for this particular comparison — Step 6 shows
exactly why that won't always be true.)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(),
}

for name, model in candidate_models.items():
    model.fit(X_train, y_train)                       # identical call, every time
    accuracy = accuracy_score(y_test, model.predict(X_test))   # identical call, every time
    print(f"{name:<24} accuracy = {accuracy:.3f}")

Five genuinely different algorithms — a linear model, a distance-based method, a
single tree, an ensemble of trees, and a support vector machine — trained and evaluated with
exactly the same three lines of code each. That consistency is the whole point of the
Estimator API: it turns "which algorithm should I try?" into a cheap experiment instead of a
rewrite.


## Step 6: Why we need preprocessing — feature scaling

Distance-based algorithms like k-NN compare *raw numeric distances* between features. If one
feature is measured in units that happen to produce much bigger numbers than another, it will
dominate that distance calculation — **even if it's actually less useful for the
prediction**. Let's build a deliberately extreme example to see this go wrong, then fix it.

We'll simulate two features for a yes/no classification task:
- **"income"** (dollars, roughly 20,000-80,000) — numerically huge, but only *barely*
  different between the two classes (not very useful on its own)
- **"years of experience"** (roughly 0-12) — numerically tiny, but *strongly* different
  between the two classes (the real signal)


In [ ]:
rng = np.random.default_rng(seed=5)
n_per_class = 150

# "income": huge numeric range, but almost identical between classes - not very informative
income_class0 = rng.normal(50000, 15000, n_per_class)
income_class1 = rng.normal(52000, 15000, n_per_class)

# "years of experience": tiny numeric range, but clearly separated between classes - the real signal
experience_class0 = rng.normal(3, 1.5, n_per_class)
experience_class1 = rng.normal(9, 1.5, n_per_class)

X_raw = np.column_stack([
    np.concatenate([income_class0, income_class1]),
    np.concatenate([experience_class0, experience_class1]),
])
y_raw = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)]).astype(int)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_raw, y_raw, test_size=0.3, random_state=5)

print("income range:     %.0f to %.0f" % (Xr_train[:, 0].min(), Xr_train[:, 0].max()))
print("experience range: %.1f to %.1f" % (Xr_train[:, 1].min(), Xr_train[:, 1].max()))

In [ ]:
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(Xr_train, yr_train)
acc_unscaled = accuracy_score(yr_test, knn_unscaled.predict(Xr_test))
print(f"k-NN accuracy WITHOUT scaling: {acc_unscaled:.3f}")
print("Income's huge numeric range swamps the distance calculation, drowning out the feature")
print("(experience) that actually separates the two classes.")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xr_train_scaled = scaler.fit_transform(Xr_train)   # fit ONLY on training data
Xr_test_scaled = scaler.transform(Xr_test)          # apply the same transform to test data

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(Xr_train_scaled, yr_train)
acc_scaled = accuracy_score(yr_test, knn_scaled.predict(Xr_test_scaled))
print(f"k-NN accuracy WITH scaling:    {acc_scaled:.3f}")
print(f"\nScaling alone turned a {acc_unscaled:.0%}-accurate model into a {acc_scaled:.0%}-accurate one,")
print("on the exact same data, with the exact same algorithm.")

`StandardScaler` rescales every feature to have mean 0 and standard deviation 1, so no
feature can dominate purely because of the units it happens to be measured in. Note that we
called `fit_transform` on the **training** data only, then `transform` (not `fit_transform`)
on the test data — fitting the scaler on test data too would leak information about the test
set into preprocessing. We'll come back to exactly this kind of leakage, in a more dangerous
form, in Step 9.


## Step 7: Training and evaluating a classifier properly

Now let's apply that same pattern — scale, then fit — to our real Iris classification task,
and look at how to actually judge whether a model is any good, beyond a single accuracy
number.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

print("Predicted:", y_pred)
print("Actual:   ", y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred))

Accuracy alone can hide problems — e.g. a model that's 95% accurate overall but
terrible at one specific class. `classification_report` breaks performance down per class
(precision, recall, F1-score), and the confusion matrix shows exactly which classes get
confused with which.


## Step 8: Why one train/test split isn't enough

`train_test_split` picks its test set randomly. That means the accuracy we measure depends
partly on *which* points happened to land in the test set — good or bad luck, not just model
quality. Let's measure that luck directly, by repeating the split many times.


In [ ]:
single_split_scores = []
for seed in range(20):
    Xs_train, Xs_test, ys_train, ys_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    model = RandomForestClassifier(random_state=42)
    model.fit(Xs_train, ys_train)
    single_split_scores.append(accuracy_score(ys_test, model.predict(Xs_test)))

single_split_scores = np.array(single_split_scores)
print("Accuracy across 20 different random splits:")
print(np.round(single_split_scores, 3))
print(f"\nMean: {single_split_scores.mean():.3f}   Std dev: {single_split_scores.std():.3f}")
print(f"Best split made it look like: {single_split_scores.max():.3f}")
print(f"Worst split made it look like: {single_split_scores.min():.3f}")
print("Same model, same data - the reported accuracy shifts just from which points got split where.")

In [ ]:
from sklearn.model_selection import cross_val_score

# cross_val_score splits the data into 5 folds, trains on 4, tests on the 5th, and repeats
# 5 times so that EVERY point gets used for testing exactly once - no single unlucky split
cv_scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
print("Cross-validation accuracy per fold:", np.round(cv_scores, 3))
print(f"Mean accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
print("\nThis single mean-and-std pair is a far more honest summary of how the model actually")
print("performs than any one train/test split above.")

## Step 9: Pipelines — bundling steps, and avoiding data leakage

A `Pipeline` chains preprocessing and modelling steps into a single estimator that still
exposes `fit`/`predict`, exactly like Step 6's manual "scale, then fit" — just as one object
instead of two separate steps to remember to keep in sync.


In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000)),
])

# cross_val_score can now cross-validate the WHOLE pipeline: for every fold, the scaler is
# fit fresh on that fold's training data only, exactly as it should be
pipeline_scores = cross_val_score(pipeline, X, y, cv=5)
print("Pipeline cross-validation accuracy:", np.round(pipeline_scores, 3))
print(f"Mean: {pipeline_scores.mean():.3f}")

That last point — fitting preprocessing fresh on each fold — matters more than it might
look. If you preprocess using statistics from the **entire** dataset *before* splitting into
folds (or into train/test), information about the test data leaks into training. This is
called **data leakage**, and it can inflate reported accuracy dramatically. Let's manufacture
an extreme, unmistakable example: 500 completely random, meaningless features, and completely
random labels with **no real relationship to the data whatsoever**. True accuracy should sit
right around 50% — pure chance.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(seed=9)
n_samples, n_noise_features = 200, 500
X_noise = rng.normal(size=(n_samples, n_noise_features))
y_noise = rng.integers(0, 2, size=n_samples)   # pure random labels - X_noise tells us NOTHING real

# --- THE LEAKY WAY: pick the "best" 10 features using the WHOLE dataset first ---
selector_leaky = SelectKBest(f_classif, k=10)
X_selected_leaky = selector_leaky.fit_transform(X_noise, y_noise)   # sees every label, incl. test rows

leaky_scores = cross_val_score(LogisticRegression(max_iter=1000), X_selected_leaky, y_noise, cv=5)
print("LEAKY approach (select features on the full dataset, THEN cross-validate):")
print(f"  Reported accuracy: {leaky_scores.mean():.3f}   <- looks far better than random guessing!")

In [ ]:
# --- THE CORRECT WAY: feature selection happens INSIDE the cross-validation pipeline ---
correct_pipeline = Pipeline([
    ("select", SelectKBest(f_classif, k=10)),
    ("classifier", LogisticRegression(max_iter=1000)),
])

correct_scores = cross_val_score(correct_pipeline, X_noise, y_noise, cv=5)
print("CORRECT approach (feature selection refit on each training fold only):")
print(f"  Reported accuracy: {correct_scores.mean():.3f}   <- correctly close to 50% random guessing")

The leaky version cherry-picks features that happen to correlate with the labels
across the *entire* dataset — including the rows that later get used for "testing" — so of
course it looks predictive; it's cheating. The pipeline version selects features fresh inside
each training fold, exactly the way `Pipeline` handled scaling above, and correctly reports
that this data is unpredictable. This is the single biggest practical reason `Pipeline` exists:
it makes the leak-free way the *easy*, default way to write the code.


## Step 10: Hyperparameter tuning — manual search vs. `GridSearchCV`

Some settings — like how many trees a random forest builds, or how deep each tree is allowed
to grow — aren't learned from data; they must be chosen. Let's search for good values by hand
first, then check that `GridSearchCV` finds the same answer automatically.


In [ ]:
import itertools

param_grid = {"n_estimators": [50, 100, 150], "max_depth": [None, 3, 5]}
combinations = list(itertools.product(param_grid["n_estimators"], param_grid["max_depth"]))

start = time.time()
manual_results = []
for n_estimators, max_depth in combinations:
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=3)
    manual_results.append((n_estimators, max_depth, scores.mean()))
manual_time = time.time() - start

best_manual = max(manual_results, key=lambda r: r[2])
print(f"Manual search over {len(combinations)} combinations took {manual_time:.2f}s")
print(f"Best: n_estimators={best_manual[0]}, max_depth={best_manual[1]}, cv accuracy={best_manual[2]:.3f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

start = time.time()
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=param_grid, cv=3)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

print(f"GridSearchCV over the same {len(combinations)} combinations took {grid_time:.2f}s")
print(f"Best: {grid_search.best_params_}, cv accuracy={grid_search.best_score_:.3f}")
print(f"\nSame best score found: {best_manual[2]:.3f} (manual) vs {grid_search.best_score_:.3f} (GridSearchCV)")

Same result, far less code — and `GridSearchCV` also parallelises the search across
CPU cores automatically (`n_jobs=-1`), and automatically refits the best-performing model on
the full training set when it's done, ready to use with `.predict()`. For larger searches,
`RandomizedSearchCV` samples a fixed number of combinations instead of trying every single one,
trading a little thoroughness for a lot of speed.


## Step 11: Capstone — a real neural network, trained with scikit-learn

Scikit-learn includes its own neural network: `MLPClassifier`, a **Multi-Layer Perceptron**
trained with backpropagation — the same core algorithm every deep learning framework runs,
just without the framework's flexibility or GPU acceleration.

We'll train it on the exact same dataset used in the `NumPy_Fundamentals` capstone: **1,797
real handwritten digit images**, 8x8 pixels each. There, we built a softmax-regression network
completely from scratch — hand-written forward pass, cross-entropy loss, and gradients,
trained over 300 steps — reaching about **95.5% test accuracy**. Let's see what a one-line
`.fit()` call gets us here, and how it compares to several classical models.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_digits, y_digits = digits.data, digits.target   # already flattened: shape (1797, 64)

print("X_digits shape:", X_digits.shape)
print("y_digits shape:", y_digits.shape)

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=1, stratify=y_digits
)
print("Training images:", Xd_train.shape[0], " Test images:", Xd_test.shape[0])

In [ ]:
import matplotlib.pyplot as plt

example_indices = np.random.default_rng(seed=3).choice(len(Xd_train), size=10, replace=False)

plt.figure(figsize=(10, 3))
for i, index in enumerate(example_indices):
    plt.subplot(2, 5, i + 1)
    plt.imshow(Xd_train[index].reshape(8, 8), cmap="gray")
    plt.title(f"Label: {yd_train[index]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.neural_network import MLPClassifier

candidate_pipelines = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))]),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF kernel)": Pipeline([("scale", StandardScaler()), ("clf", SVC())]),
    "Neural Network (MLPClassifier)": Pipeline([
        ("scale", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=42)),
    ]),
}

results = {}
for name, model in candidate_pipelines.items():
    start = time.time()
    model.fit(Xd_train, yd_train)
    train_time = time.time() - start
    accuracy = accuracy_score(yd_test, model.predict(Xd_test))
    results[name] = accuracy
    print(f"{name:<32} accuracy={accuracy:.4f}   train_time={train_time:.2f}s")

print(f"\nFor reference, the NumPy-from-scratch softmax regression reached: 0.9547")

Every model here is trained in a single `.fit()` call, using the exact same three
lines of code as Step 5 — the same interface that let us swap k-NN for a decision tree also
lets us swap in a neural network, with no special-case code required.

`MLPClassifier` also tracks its training loss at every iteration, exactly like the loss curve
we plotted by hand in `NumPy_Fundamentals` — except here it was recorded automatically.


In [ ]:
mlp = candidate_pipelines["Neural Network (MLPClassifier)"].named_steps["clf"]

plt.figure(figsize=(6, 4))
plt.plot(mlp.loss_curve_)
plt.xlabel("Training iteration")
plt.ylabel("Loss")
plt.title("MLPClassifier training loss curve")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
mlp_predictions = candidate_pipelines["Neural Network (MLPClassifier)"].predict(Xd_test)
example_indices = np.random.default_rng(seed=4).choice(len(Xd_test), size=8, replace=False)

plt.figure(figsize=(12, 5))
for i, index in enumerate(example_indices):
    plt.subplot(2, 4, i + 1)
    plt.imshow(Xd_test[index].reshape(8, 8), cmap="gray")
    plt.axis("off")
    predicted, actual = mlp_predictions[index], yd_test[index]
    color = "green" if predicted == actual else "red"
    plt.title(f"Pred: {predicted}  True: {actual}", color=color)
plt.tight_layout()
plt.show()

### `MLPClassifier`'s limits

`MLPClassifier` is genuinely useful for tabular data and for learning the concepts, but it's
not a substitute for a dedicated deep learning framework:

| | scikit-learn `MLPClassifier` | TensorFlow/Keras, PyTorch |
|---|---|---|
| Architecture flexibility | Fixed feedforward layers only | Any architecture (CNNs, RNNs, Transformers, custom layers) |
| GPU acceleration | No | Yes |
| Scale | Small/medium datasets | Large-scale, distributed training |
| API complexity | Very simple (`fit`/`predict`) | More code, more control |
| Best for | Quick prototypes, tabular data, learning | Images, text, audio, custom architectures, production-scale deep learning |


## Step 12: How scikit-learn fits alongside deep learning frameworks

Even when the final model is a neural network built in TensorFlow/Keras or PyTorch,
scikit-learn is usually still doing the work *around* it:

- **`train_test_split` / `KFold`** — splitting data before it ever reaches the network
- **`StandardScaler`, `OneHotEncoder`, `SelectKBest`** — preprocessing features and labels
  into the numeric form a neural network needs (exactly what Steps 6 and 9 covered)
- **`Pipeline` / `ColumnTransformer`** — reproducible preprocessing for tabular data before it's
  batched into a network
- **`sklearn.metrics`** — accuracy, precision/recall, ROC-AUC, confusion matrices, computed on
  a deep learning model's predictions just as in Step 7
- **`GridSearchCV` / `RandomizedSearchCV`** — sometimes wrapped around a Keras model via an
  adapter like **SciKeras**, so a neural network's hyperparameters can be tuned with the exact
  tools used on `RandomForestClassifier` in Step 10
- **A quick baseline** — a `LogisticRegression` or `RandomForestClassifier`, trained in one
  line as in Step 11, is often the first thing built to prove a neural network is actually
  worth its extra complexity

The illustrative snippet below (not executed here, since it needs `tensorflow` installed)
shows this division of labour on the same digits data from Step 11: scikit-learn handles
splitting and evaluation, Keras defines and trains the network itself.

```python
# --- Illustrative only: requires `pip install tensorflow` ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow import keras

# 1) scikit-learn: split and scale the data
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_digits, y_digits, test_size=0.2, random_state=1)
scaler = StandardScaler()
Xd_train_scaled = scaler.fit_transform(Xd_train)
Xd_test_scaled = scaler.transform(Xd_test)

# 2) Keras: define and train the neural network
model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(64,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(Xd_train_scaled, yd_train, epochs=50, verbose=0)

# 3) scikit-learn: evaluate with standard metrics
y_pred = model.predict(Xd_test_scaled).argmax(axis=1)
print(classification_report(yd_test, y_pred))
```


## Summary

This notebook built up scikit-learn understanding from first principles to a trained neural
network:

1. **Writing ML algorithms by hand is tedious and error-prone** — even k-NN needs real care;
   scikit-learn gives tested, optimised implementations instead (Step 2)
2. **Every estimator shares the same `fit`/`predict`/`transform` interface** (Step 3)
3. Data is loaded and split with `train_test_split` before any model sees it (Step 4)
4. That shared interface means **swapping algorithms costs one line, not a rewrite** (Step 5)
5. **Feature scaling matters** for distance-based models — we watched an identical model go
   from misleading to accurate purely by rescaling its inputs (Step 6)
6. **`classification_report` and confusion matrices** reveal what a single accuracy number
   hides (Step 7)
7. **A single train/test split is noisy** — cross-validation reports a far more honest,
   stable estimate of performance (Step 8)
8. **`Pipeline`** bundles preprocessing and modelling into one estimator, and — critically —
   prevents data leakage; we watched a leaky pipeline claim ~80%+ "accuracy" on pure random
   noise, while the correct pipeline correctly reported ~50% (Step 9)
9. **`GridSearchCV` automates hyperparameter search** — we verified it finds exactly the same
   best answer as a hand-written nested loop, with far less code (Step 10)
10. We trained scikit-learn's own neural network, **`MLPClassifier`**, on real handwritten
    digit images — the same dataset the `NumPy_Fundamentals` notebook trained a from-scratch
    softmax regression network on — and compared it against several classical models side by
    side (Step 11)
11. Even outside scikit-learn's own models, its **preprocessing, splitting, metrics, and
    hyperparameter search tools** are used constantly around deep learning frameworks like
    TensorFlow/Keras and PyTorch (Step 12)

### Key terms

- **Estimator**: any scikit-learn object with a `fit` method; the common interface behind
  every model, scaler, and selector in the library.
- **Transformer**: an estimator with a `transform` method (e.g. `StandardScaler`,
  `SelectKBest`) that converts data rather than predicting a label.
- **Pipeline**: chains preprocessing and modelling steps into a single estimator, so they're
  always applied together, in the same order, on both training and new data.
- **Cross-validation**: repeatedly splitting data into different train/test folds and
  averaging the result, to get a stable performance estimate instead of relying on one split.
- **Data leakage**: information from data that should be "unseen" (like the test set)
  accidentally influencing training — often through preprocessing fit on the full dataset
  before splitting — which makes reported performance look better than it really is.
- **Hyperparameter**: a setting chosen before training (like a forest's number of trees)
  rather than learned from the data itself.
- **`GridSearchCV`**: exhaustively tries every combination in a hyperparameter grid, using
  cross-validation to score each one, and keeps the best.
- **MLP (Multi-Layer Perceptron)**: a basic feedforward neural network — layers of weights and
  biases connected by activation functions, trained with backpropagation.

### How this connects to the NumPy Fundamentals notebook

`NumPy_Fundamentals` built a neural network's forward pass, loss function, and gradient
descent loop entirely by hand, to show *what's actually happening* inside a model. This
notebook shows the other half of the picture: how scikit-learn takes that same underlying
maths — plus dozens of other algorithms, proper evaluation, and safe preprocessing — and puts
it behind a small, consistent, well-tested API, so you rarely need to write it by hand again.


## Ideas to extend

- Use `RandomizedSearchCV` instead of `GridSearchCV` on a much larger hyperparameter grid, and
  compare how close it gets to the true best score in a fraction of the time
- Try `ColumnTransformer` on a dataset with a mix of numeric and categorical features, applying
  `StandardScaler` and `OneHotEncoder` to different columns within a single pipeline
- Install `tensorflow` and actually run Step 12's illustrative snippet, then compare its
  accuracy against Step 11's `MLPClassifier` on the same digits data
- Look up **SciKeras**, which wraps a Keras model so it can be tuned with `GridSearchCV`
  exactly like `RandomForestClassifier` was in Step 10
- Save a trained pipeline with `joblib.dump`/`joblib.load` and reload it in a fresh Python
  session, to see how a trained scikit-learn model is normally persisted for later use
- Rerun Step 11 on the full-resolution, 70,000-image MNIST dataset (e.g. via
  `sklearn.datasets.fetch_openml('mnist_784')`) and compare training time and accuracy against
  the smaller 8x8 digits dataset used here
